<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# 07 · Automatización de Excel con openpyxl (I)

Seguimos con los diez documentos del observatorio del bloque 06. Leeremos un XLSX de entrada y crearemos una entrega con Documentos y Resumen. Las decisiones documentales se conservan; Excel es una vista de seguimiento, no un nuevo clasificador.

## Requisitos y trabajo

Python 3.11+, openpyxl 3.1.5 y Jupyter solo para esta interfaz. Inicia Jupyter desde la carpeta del bloque y conserva los tres motores Python juntos. Instala requirements.txt en el mismo intérprete. Ejecuta desde un kernel limpio y en orden.

La práctica crea demos temporales propias; no registra tareas ni abre documentos personales. No repitas una celda que crea una raíz nueva sin reiniciar la sesión. Cada ejercicio tiene una bandera completado_NN: actívala tras terminar tu respuesta. Las salidas del solucionario pertenecen a una ejecución de comprobación y sus rutas no se deben reutilizar como configuración.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import date, datetime
from collections import Counter
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import sys, json, hashlib
PROYECTO=Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (PROYECTO/"excel_observatorio.py").is_file():
    raise FileNotFoundError("Inicia Jupyter en la carpeta del bloque")
sys.path.insert(0,str(PROYECTO))
import excel_observatorio as e
temporal=TemporaryDirectory(prefix="excel07_aula_")
base=Path(temporal.name)
def caso(n):
    return e.preparar_demo(base/f"caso_{n:02d}")
print("Contenedor propio:",base)


## 1 · Modelo de objetos

Workbook contiene Worksheet; Worksheet contiene Cell. El acceso por nombre expresa el contrato. No se crea archivo al construir un libro en memoria.

In [ ]:
guia_wb=Workbook()
guia_ws=guia_wb.active
guia_ws.title="Documentos"
guia_wb.create_sheet("Resumen",0)
print(guia_wb.sheetnames)


## 2 · Coordenadas y valores

row y column empiezan en uno. Una tupla de iter_rows usa índices Python desde cero. value y data_type describen contenido; number_format describe apariencia.

In [ ]:
guia_ws["A1"]="entrada_id"
guia_ws.cell(row=2,column=1,value="0001")
guia_ws["A2"].number_format="@"
print(guia_ws["A2"].coordinate,guia_ws["A2"].value,guia_ws["A2"].data_type)


## 3 · Número, fecha y ausencia

La fecha se escribe con date, el tamaño con int y la ausencia con None. Un cero numérico no es ausencia. Una cadena parecida a una fecha sigue siendo texto.

In [ ]:
guia_ws["B2"]=date(2026,10,1)
guia_ws["B2"].number_format="dd/mm/yyyy"
guia_ws["C2"]=0
guia_ws["D2"]=None
print(type(guia_ws["B2"].value).__name__,guia_ws["C2"].value,guia_ws["D2"].value)


## 4 · Recorridos acotados

values_only=True devuelve valores; sin esta opción se conservan objetos Cell para inspeccionar tipos. Una celda lejana o con formato puede ampliar dimensiones aparentes.

In [ ]:
for fila in guia_ws.iter_rows(min_row=1,max_row=2,min_col=1,max_col=4,values_only=True):
    print(fila)


## 5 · Presentación funcional

Asignar objetos Font, PatternFill y Alignment. Los componentes de estilo asignados no se mutan directamente: asignar uno nuevo o copiar antes. Un formato no corrige un dato de tipo incorrecto.

In [ ]:
guia_ws["A1"].font=Font(bold=True,color="FFFFFF")
guia_ws["A1"].fill=PatternFill("solid",fgColor="003B5C")
guia_ws.column_dimensions["A"].width=18
guia_ws.freeze_panes="B2"
guia_ws.auto_filter.ref="A1:D2"
print(guia_ws.freeze_panes,guia_ws.auto_filter.ref)


## 6 · Guardar y reabrir

Workbook.save puede sobrescribir: aquí usamos un nombre dentro del contenedor propio. La entrega integrada emplea guardar_nuevo, contraste y publicación desde un parcial.

In [ ]:
guia_ruta=base/"guia.xlsx"
guia_wb.save(guia_ruta)
guia_wb.close()
guia_leido=load_workbook(guia_ruta)
try:
    print(guia_leido.sheetnames,type(guia_leido["Documentos"]["B2"].value).__name__)
finally:
    guia_leido.close()


## 7 · Entradas del mismo observatorio

La preparación reutiliza organizador.py del bloque 06. La primera aplicación conserva los estados creado 4, duplicado_lote 1, revision 3 y conflicto_lote 2. Cada fila es una entrada física.

In [ ]:
guia_proyecto=caso(0)
guia_filas=e.leer_entradas(guia_proyecto/"entrada_excel/seguimiento_entrada.xlsx")
print(len(guia_filas),Counter(f["estado"] for f in guia_filas))
print("Fechas ausentes:",sum(f["fecha"] is None for f in guia_filas))


## 8 · Simulación y publicación

generar valida sin escribir por defecto. Al aplicar, crea una entrega en salida_excel y la verifica. Los recuentos de Resumen son una instantánea de Python, no fórmulas que reaccionan a ediciones manuales.

In [ ]:
guia_sim=e.generar(guia_proyecto)
print(guia_sim)
assert not list((guia_proyecto/"salida_excel").iterdir())


## Ejercicio 01 · Libro y hojas

Crea wb1 con hoja Documentos y Resumen en primera posición. Recupera Documentos por nombre.

**Pista:** Workbook(), active.title y create_sheet(...,0).

In [ ]:
# TODO: completa la actividad descrita.

completado_01=False


In [ ]:
if completado_01:
    assert wb1.sheetnames==["Resumen","Documentos"] and ws1.title=="Documentos"
    print("Ejercicio 01: OK")
else:
    print("Ejercicio 01: pendiente")


**Interpretación:** Elegir la hoja por nombre evita depender de una selección activa que otra persona puede cambiar.

## Ejercicio 02 · Identidad frente a apariencia

Escribe texto "0001" en A1 y número 1 con formato "0000" en B1. Comprueba que sus valores y tipos son distintos.

**Pista:** data_type será s para texto y n para número.

In [ ]:
# TODO: completa la actividad descrita.

completado_02=False


In [ ]:
if completado_02:
    assert ws2["A1"].value=="0001" and ws2["A1"].data_type=="s" and ws2["B1"].value==1
    print("Ejercicio 02: OK")
else:
    print("Ejercicio 02: pendiente")


**Interpretación:** Un formato no recupera ceros iniciales perdidos. El tipo se decide por significado del campo.

## Ejercicio 03 · Fecha, cero y ausencia

Crea un libro con fecha en A1, cero en B1 y None en C1. Guarda en base, reabre y contrasta.

**Pista:** Al cargar una fecha puede obtenerse datetime. Normalizar con .date() si procede.

In [ ]:
# TODO: completa la actividad descrita.

completado_03=False


In [ ]:
if completado_03:
    assert fecha3==date(2026,10,1) and cero3==0 and ausente3 is None
    print("Ejercicio 03: OK")
else:
    print("Ejercicio 03: pendiente")


**Interpretación:** La ausencia y el cero representan situaciones distintas. None permite no inventar una fecha para un nombre inválido.

## Ejercicio 04 · Leer las diez entradas

Crea p4 con caso(4) y utiliza leer_entradas para recuperar datos. Cuenta IDs únicos y fechas ausentes.

**Pista:** La hoja de entrada se llama Entrada.

In [ ]:
# TODO: completa la actividad descrita.

completado_04=False


In [ ]:
if completado_04:
    assert len(f4)==10 and len({f["entrada_id"] for f in f4})==10 and sum(f["fecha"] is None for f in f4)==2
    print("Ejercicio 04: OK")
else:
    print("Ejercicio 04: pendiente")


**Interpretación:** No se deduplican las entradas físicas. Las dos versiones contradictorias y la repetición siguen presentes en el seguimiento.

## Ejercicio 05 · Rechazar cabeceras y fórmulas

Crea p5. En su XLSX propio, cambia cabecera A1; comprueba rechazo. Restaura los bytes originales y escribe una fórmula en D2; vuelve a comprobar rechazo.

**Pista:** Guardar los bytes originales permite restaurar solo el archivo sintético del caso.

In [ ]:
# TODO: completa la actividad descrita.

completado_05=False


In [ ]:
if completado_05:
    assert rechazos5==2
    print("Ejercicio 05: OK")
else:
    print("Ejercicio 05: pendiente")


**Interpretación:** data_only=False permite detectar la fórmula en la entrada. data_only=True podría ocultarla tras un valor cacheado o None.

## Ejercicio 06 · Resumen reconciliado

Crea p6, lee filas y calcula conteo y pendientes documentales.

**Pista:** Counter sobre estado; sumar revision, conflicto_lote y conflicto_destino.

In [ ]:
# TODO: completa la actividad descrita.

completado_06=False


In [ ]:
if completado_06:
    assert sum(conteo6.values())==10 and conteo6["creado"]==4 and pendientes6==5
    print("Ejercicio 06: OK")
else:
    print("Ejercicio 06: pendiente")


**Interpretación:** El total es diez entradas, no cuatro salidas. El duplicado no aumenta el número de publicaciones.

## Ejercicio 07 · Construir las dos hojas

Crea p7, lee sus filas y construye el libro con crear_libro. Inspecciona hojas, fila de cabeceras y total.

**Pista:** Datos de Documentos empiezan en fila 5.

In [ ]:
# TODO: completa la actividad descrita.

completado_07=False


In [ ]:
if completado_07:
    assert w7.sheetnames==["Resumen","Documentos"] and w7["Documentos"].max_row==14 and w7["Resumen"]["B13"].value==10
    print("Ejercicio 07: OK")
else:
    print("Ejercicio 07: pendiente")


**Interpretación:** El libro se construye en memoria. Los valores del resumen son recuentos Python: una edición posterior en Excel no los recalcula.

## Ejercicio 08 · Formato y navegación

Crea p8 y su libro. Comprueba formato de C5, D5, paneles y filtro de Documentos.

**Pista:** dd/mm/yyyy, #,##0, C5 y A4:F14.

In [ ]:
# TODO: completa la actividad descrita.

completado_08=False


In [ ]:
if completado_08:
    assert ws8["C5"].number_format=="dd/mm/yyyy" and ws8["D5"].number_format=="#,##0"
    assert ws8.freeze_panes=="C5" and ws8.auto_filter.ref=="A4:F14"
    print("Ejercicio 08: OK")
else:
    print("Ejercicio 08: pendiente")


**Interpretación:** C5 inmoviliza filas uno a cuatro y columnas A y B. auto_filter configura controles de Excel; no elimina registros en Python.

## Ejercicio 09 · Publicar y verificar

Crea p9, conserva el hash del libro de entrada y genera la entrega aplicando. Verifica sus filas y que el original no cambió.

**Pista:** guardar_nuevo lo usa el motor generar.

In [ ]:
# TODO: completa la actividad descrita.

completado_09=False


In [ ]:
if completado_09:
    assert r9["verificado"] and hashlib.sha256(entrada9.read_bytes()).hexdigest()==hash9
    print("Ejercicio 09: OK")
else:
    print("Ejercicio 09: pendiente")


**Interpretación:** Reabrir y comparar contenido es más informativo que comprobar solo que el archivo existe.

## Ejercicio 10 · No sobrescribir una entrega

Crea p10, genera una entrega, guarda sus bytes e intenta generarla de nuevo con el mismo nombre. Captura el rechazo.

**Pista:** FileExistsError; para otra versión deliberada usar nombre distinto.

In [ ]:
# TODO: completa la actividad descrita.

completado_10=False


In [ ]:
if completado_10:
    assert rechazo10 and salida10.read_bytes()==antes10
    print("Ejercicio 10: OK")
else:
    print("Ejercicio 10: pendiente")


**Interpretación:** La política de publicación de entregas es distinta de una actualización sobre el mismo archivo. No sobrescribir por accidente al repetir una celda.

## Ejercicio 11 · Fórmula y caché

Crea B1=4, B2=6 y B3="=SUM(B1:B2)" en un libro nuevo. Guarda y lee con data_only False y True.

**Pista:** El libro no pasa por un motor de cálculo: no habrá caché calculada.

In [ ]:
# TODO: completa la actividad descrita.

completado_11=False


In [ ]:
if completado_11:
    assert formula11=="=SUM(B1:B2)" and cache11 is None
    print("Ejercicio 11: OK")
else:
    print("Ejercicio 11: pendiente")


**Interpretación:** El resultado matemático esperado es diez, pero openpyxl no lo calcula. None en la caché de este libro nuevo no es un error de sintaxis de la fórmula.

## Ejercicio 12 · Texto literal con signo igual

Usa escribir_celda para guardar "=1+1" como motivo literal en un libro aislado. Reabre y comprueba texto y tipo.

**Pista:** El helper fuerza data_type="s" para strings.

In [ ]:
# TODO: completa la actividad descrita.

completado_12=False


In [ ]:
if completado_12:
    assert texto12=="=1+1" and tipo12=="s"
    print("Ejercicio 12: OK")
else:
    print("Ejercicio 12: pendiente")


**Interpretación:** Los datos externos se preservan como datos. Las fórmulas se escriben solo cuando el flujo lo exige de manera deliberada.

## Cierre y evaluación

Explica qué representa una fila, por qué se mantienen duplicados y conflictos, qué cambia el formato y cómo sabes que se conservó la entrada. Contrasta fórmula guardada y cálculo realizado. El bloque II desarrollará tablas, gráficos, validaciones y fórmulas más elaboradas. Fuentes primarias en la presentación y README.

In [ ]:
retirar_temporal=False
if retirar_temporal:
    temporal.cleanup()
else:
    print("Datos de esta sesión:",base)
